# we want to recommend similar products based on what people say in reviews(content-based filtering)
Step 1: convert each review's text → a vector (via sentence-transformers)

Step 2: average the vectors for all reviews of one product → one "meaning fingerprint" per product

Step 3: load all product vectors into FAISS

Step 4: given one product, ask FAISS "which other vectors are closest?" → those become your recommendations

In [1]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

/Users/preethapallavi/Documents/personal_project/amazon_review/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_clean = pd.read_parquet('amazon_reviews_clean.parquet')
df_clean.head()

,Id,ProductId,UserId,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,word_count
0,5,B006K2ZZ7K,A1UQRSCLF8GW1T,0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...,27
1,6,B006K2ZZ7K,ADT0SRK1MGOEU,0,0,4,1342051200,Nice Taffy,I got a wild hair for taffy and ordered this f...,72
2,7,B006K2ZZ7K,A1SP2KVKFXXRU1,0,0,5,1340150400,Great! Just as good as the expensive brands!,This saltwater taffy had great flavors and was...,49
3,8,B006K2ZZ7K,A3JRGQVEQN31IQ,0,0,5,1336003200,"Wonderful, tasty taffy",This taffy is so good. It is very soft and ch...,24
4,14,B001GVISJM,A18ECVX2RJ7HUE,2,2,4,1288915200,fresh and greasy!,good flavor! these came securely packed... the...,15


In [3]:
# 1. Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Embed every review (batched, not row-by-row)
embeddings = model.encode(df_clean['Text'].tolist(), batch_size=64, show_progress_bar=True)
df_clean['embedding'] = list(embeddings)

Batches: 100%|██████████| 5347/5347 [21:05<00:00,  4.22it/s]  


In [4]:
# 3. Mean-pool embeddings per product
product_vectors = df_clean.groupby('ProductId')['embedding'].apply(lambda x: np.mean(np.vstack(x), axis=0))
product_ids = product_vectors.index.tolist()
matrix = np.vstack(product_vectors.values).astype('float32')

# 4. Normalize (needed for cosine similarity via inner product)
faiss.normalize_L2(matrix)

# 5. Build FAISS index
index = faiss.IndexFlatIP(matrix.shape[1])
index.add(matrix)

print("Products indexed:", index.ntotal)


Products indexed: 27606


In [21]:
# 6. Test — get top 5 similar products for the first product
query_vec = matrix[0:1]
D, I = index.search(query_vec, k=5)

for rank, (idx, score) in enumerate(zip(I[0], D[0])):
    print(rank, product_ids[idx], round(float(score), 3))

0 0006641040 1.0
1 B001ID8A28 0.799
2 B000ILIJRM 0.562
3 B0007XAUXC 0.549
4 B0009ORSRK 0.519


In [20]:
def inspect_product(pid, n=3):
    print(f"--- {pid} ---")
    print(df_clean[df_clean['ProductId'] == pid]['Text'].head(n).to_string(index=False))
    print()

# the query product
inspect_product('0006641040')

# its top recommendations
for idx in I[0][1:]:  # skip index 0 (itself)
    inspect_product(product_ids[idx])

--- 0006641040 ---
These days, when a person says, "chicken soup" they're probably going to follow up those words with, "for the soul" or maybe "for the teenaged soul".  Didn't used to be that way.  Why I can remember a time when if a person said, "chicken soup" those words were followed by an enthusiastic "with rice!".  Such was the power of Maurice Sendak's catchy 1962 children's book.  I am pleased to report that if you care to read this book again today, you will find it hasn't dimished a jot in terms of frolicksome fun.  In this book we are led through a whirlwind chicken soup year with our host, a boy who bears no little resemblance to Sendak's other great rhyming tale "Pierre" (in looks if not demeanor).  It's a catchy flouncy bouncy combo of soup and the people who love it so.  This is ostensibly a book meant to teach your children the different months of the year.  Each month gets its own rhythmic poem and accompanying illustration.  These are fairly simple pen and ink drawing

In [19]:
sample_ids = ['B001ID8A28', 'B000ILIJRM']
for pid in sample_ids:
    idx_pos = product_ids.index(pid)
    query_vec = matrix[idx_pos:idx_pos+1]
    D, I = index.search(query_vec, k=4)
    print(f"Query: {pid}")
    for i in I[0][1:]:
        inspect_product(product_ids[i], n=1)

Query: B001ID8A28
--- 0006641040 ---
These days, when a person says, "chicken soup" they're probably going to follow up those words with, "for the soul" or maybe "for the teenaged soul".  Didn't used to be that way.  Why I can remember a time when if a person said, "chicken soup" those words were followed by an enthusiastic "with rice!".  Such was the power of Maurice Sendak's catchy 1962 children's book.  I am pleased to report that if you care to read this book again today, you will find it hasn't dimished a jot in terms of frolicksome fun.  In this book we are led through a whirlwind chicken soup year with our host, a boy who bears no little resemblance to Sendak's other great rhyming tale "Pierre" (in looks if not demeanor).  It's a catchy flouncy bouncy combo of soup and the people who love it so.  This is ostensibly a book meant to teach your children the different months of the year.  Each month gets its own rhythmic poem and accompanying illustration.  These are fairly simple p

# save

In [24]:
import faiss, pickle

faiss.write_index(index, 'product_index.faiss')

with open('product_ids.pkl', 'wb') as f:
    pickle.dump(product_ids, f)

# small lookup table: product -> one sample review (for display purposes)
product_sample_text = df_clean.drop_duplicates('ProductId').set_index('ProductId')['Text'].to_dict()
with open('product_sample_text.pkl', 'wb') as f:
    pickle.dump(product_sample_text, f)

In [22]:
import random
from collections import defaultdict

# 1. Build user -> list of products they reviewed (only users with 2+ products)
user_products = df_clean.groupby('UserId')['ProductId'].apply(set)
user_products = user_products[user_products.apply(len) >= 2]

print("Users with 2+ reviewed products:", len(user_products))

# 2. Build a quick lookup: ProductId -> its position in matrix
pid_to_pos = {pid: i for i, pid in enumerate(product_ids)}

# 3. Evaluate Recall@k
def evaluate_recall_at_k(k=10, sample_size=500):
    random.seed(42)
    users_sample = random.sample(list(user_products.index), min(sample_size, len(user_products)))

    hits = 0
    total = 0

    for user in users_sample:
        products = list(user_products[user])
        # only use products that exist in our index (post-filtering may have dropped some)
        products = [p for p in products if p in pid_to_pos]
        if len(products) < 2:
            continue

        query_pid = products[0]
        relevant = set(products[1:])  # the "ground truth" — other products this user liked

        idx_pos = pid_to_pos[query_pid]
        query_vec = matrix[idx_pos:idx_pos+1]
        D, I = index.search(query_vec, k=k+1)  # +1 to account for self-match

        recommended = set(product_ids[i] for i in I[0] if product_ids[i] != query_pid)

        if recommended & relevant:  # at least one hit
            hits += 1
        total += 1

    return hits / total if total > 0 else 0

recall = evaluate_recall_at_k(k=10, sample_size=500)
print(f"Recall@10: {recall:.3f}")

Users with 2+ reviewed products: 42635
Recall@10: 0.142


In [23]:
for k in [5, 10, 20]:
    r = evaluate_recall_at_k(k=k, sample_size=500)
    print(f"Recall@{k}: {r:.3f}")

Recall@5: 0.106
Recall@10: 0.142
Recall@20: 0.176


http://127.0.0.1:8000/docs#/default/recommend_recommend__product_id__get